# EcoSignal Active Sampling Planner v1 — Real Map Data

This notebook uses the same **real GPS-aligned inputs** as `GeoRisk3D`:

- `Dem_2015.tif`
- `Dem_2022.tif`
- `DemDif_2015-2022.tif`
- `lichen_points.csv`

It rebuilds the aligned EcoSignal risk layers, generates candidate field locations from the real raster grid, ranks their scientific value, enforces geographic spacing, explains every recommendation, and exports CSV + GeoJSON files for the EcoSignal frontend.

No synthetic observations are generated or used.

> Scientific boundary: model uncertainty in v1 is an explicit **sampling-priority heuristic**, combining distance from observations, disagreement between geomorphology and lichen signals, and local CV confidence. It is not yet a statistically calibrated prediction interval.


In [ ]:
# Colab setup
import importlib.util, subprocess, sys

missing = [p for p in ("rasterio", "scipy", "pyproj") if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

import json, math, warnings
from dataclasses import dataclass, asdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
from rasterio.windows import from_bounds, Window
from rasterio.warp import transform as warp_transform, transform_bounds
from scipy.ndimage import gaussian_filter, uniform_filter, distance_transform_edt
from IPython.display import display

pd.set_option('display.max_colwidth', 120)
print('Environment ready.')


## 1. Configuration

The defaults match the folder and filenames in your GeoRisk3D notebook. Planner weights are deliberately visible and must total 1.0.


In [ ]:
@dataclass(frozen=True)
class PlannerConfig:
    dem_dir: str = "/content/drive/MyDrive/EcoSignal_DEM"
    crop_buffer_m: float = 500.0
    dem_limit_of_detection_m: float = 0.15
    lichen_smoothing_sigma_px: float = 60.0
    candidate_spacing_m: float = 30.0
    exclude_near_sample_m: float = 30.0
    minimum_selected_spacing_m: float = 120.0
    distance_uncertainty_cap_m: float = 750.0
    recommendations: int = 10
    minimum_controls: int = 2
    minimum_hotspots: int = 3
    minimum_uncertainty_checks: int = 2
    risk_weight: float = 0.28
    uncertainty_weight: float = 0.24
    sparsity_weight: float = 0.18
    recent_change_weight: float = 0.15
    control_weight: float = 0.10
    terrain_diversity_weight: float = 0.05

    def validate(self):
        weights = [self.risk_weight, self.uncertainty_weight, self.sparsity_weight,
                   self.recent_change_weight, self.control_weight, self.terrain_diversity_weight]
        if not np.isclose(sum(weights), 1.0):
            raise ValueError(f"Planner weights must total 1.0; received {sum(weights):.4f}")
        if self.recommendations < 1 or self.minimum_selected_spacing_m <= 0:
            raise ValueError("Recommendation count and spacing must be positive.")
        return self

CONFIG = PlannerConfig().validate()
DEM_DIR = Path(CONFIG.dem_dir)
PATHS = {
    "dem_2015": DEM_DIR / "Dem_2015.tif",
    "dem_2022": DEM_DIR / "Dem_2022.tif",
    "dem_difference": DEM_DIR / "DemDif_2015-2022.tif",
    "lichen_points": DEM_DIR / "lichen_points.csv",
}

display(pd.Series(asdict(CONFIG), name='value').to_frame())
print('\nInputs:')
for name, path in PATHS.items():
    print(f"- {name}: {path}")


## 2. Load and validate the real inputs

The planner fails loudly if a file or required field is missing. It also requires a projected DEM in meters so distance and spacing rules remain scientifically meaningful.


In [ ]:
missing_files = [str(path) for path in PATHS.values() if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        "Missing EcoSignal input files:\n- " + "\n- ".join(missing_files) +
        "\nUpload them to MyDrive/EcoSignal_DEM or change CONFIG.dem_dir."
    )

lichen_df = pd.read_csv(PATHS["lichen_points"])
required_columns = {
    "latitude", "longitude", "coverage_percent", "chlorosis_score",
    "nitrophyte_score", "growth_form", "cv_confidence"
}
missing_columns = required_columns - set(lichen_df.columns)
if missing_columns:
    raise ValueError(f"lichen_points.csv is missing columns: {sorted(missing_columns)}")

numeric_01 = ["chlorosis_score", "nitrophyte_score", "cv_confidence"]
for col in numeric_01:
    lichen_df[col] = pd.to_numeric(lichen_df[col], errors='coerce')
    if lichen_df[col].isna().any() or not lichen_df[col].between(0, 1).all():
        raise ValueError(f"{col} must contain numeric values from 0 to 1.")

lichen_df["coverage_percent"] = pd.to_numeric(lichen_df["coverage_percent"], errors='coerce')
if lichen_df["coverage_percent"].isna().any() or not lichen_df["coverage_percent"].between(0, 100).all():
    raise ValueError("coverage_percent must contain numeric values from 0 to 100.")
if not lichen_df.latitude.between(-90, 90).all() or not lichen_df.longitude.between(-180, 180).all():
    raise ValueError("Invalid latitude/longitude values in lichen_points.csv.")

with rasterio.open(PATHS["dem_2022"]) as src:
    if not src.crs or not src.crs.is_projected:
        raise ValueError("The DEM must use a projected CRS for meter-based distance calculations.")
    print("DEM CRS:", src.crs)
    print("DEM resolution:", src.res)

print(f"Validated {len(lichen_df)} real lichen observations.")
display(lichen_df.head())


## 3. Rebuild the GPS-aligned GeoRisk layers

This follows the real-data portion of GeoRisk3D: crop around the observations, confirm DEM-difference direction, compute slope and erosion risk, and create the confidence-weighted lichen-stress surface.


In [ ]:
def get_wgs84_bounds(df, buffer_m):
    min_lat, max_lat = df.latitude.min(), df.latitude.max()
    min_lon, max_lon = df.longitude.min(), df.longitude.max()
    mean_lat = np.deg2rad((min_lat + max_lat) / 2)
    lat_buffer = buffer_m / 111_320
    lon_buffer = buffer_m / (111_320 * np.cos(mean_lat))
    return min_lon-lon_buffer, min_lat-lat_buffer, max_lon+lon_buffer, max_lat+lat_buffer


def read_crop(path, bounds_wgs84):
    with rasterio.open(path) as src:
        raster_bounds = transform_bounds("EPSG:4326", src.crs, *bounds_wgs84, densify_pts=21)
        window = from_bounds(*raster_bounds, transform=src.transform).round_offsets().round_lengths()
        window = window.intersection(Window(0, 0, src.width, src.height))
        data = src.read(1, window=window, masked=True)
        transform = src.window_transform(window)
        profile = src.profile.copy()
        profile.update(height=data.shape[0], width=data.shape[1], transform=transform)
    return data, transform, profile


def normalize_percentile(arr, low=5, high=95):
    valid = np.isfinite(arr)
    if not valid.any():
        return np.zeros_like(arr, dtype='float32')
    p_low, p_high = np.nanpercentile(arr[valid], [low, high])
    if np.isclose(p_low, p_high):
        return np.zeros_like(arr, dtype='float32')
    return np.clip((arr-p_low)/(p_high-p_low), 0, 1).astype('float32')


crop_bounds_wgs84 = get_wgs84_bounds(lichen_df, CONFIG.crop_buffer_m)
dem_2015, transform_2015, profile_2015 = read_crop(PATHS["dem_2015"], crop_bounds_wgs84)
dem_2022, transform_2022, profile_2022 = read_crop(PATHS["dem_2022"], crop_bounds_wgs84)
dem_difference, transform_difference, profile_difference = read_crop(PATHS["dem_difference"], crop_bounds_wgs84)

if not (dem_2015.shape == dem_2022.shape == dem_difference.shape):
    raise ValueError(f"Cropped raster shapes do not align: {dem_2015.shape}, {dem_2022.shape}, {dem_difference.shape}")

d15 = dem_2015.filled(np.nan).astype('float32')
d22 = dem_2022.filled(np.nan).astype('float32')
dd_file = dem_difference.filled(np.nan).astype('float32')
dd_calc = d22 - d15
valid = np.isfinite(dd_calc) & np.isfinite(dd_file)
if valid.sum() < 2:
    raise ValueError("Not enough overlapping valid DEM pixels.")

diff_correlation = np.corrcoef(dd_calc[valid].ravel(), dd_file[valid].ravel())[0, 1]
change = -dd_file if diff_correlation < 0 else dd_file
clean_change = np.where(np.abs(change) < CONFIG.dem_limit_of_detection_m, 0, change)

xres, yres = abs(transform_2022.a), abs(transform_2022.e)
dz_dy, dz_dx = np.gradient(d22, yres, xres)
slope_2022 = np.degrees(np.arctan(np.sqrt(dz_dx**2 + dz_dy**2)))
slope_score = normalize_percentile(slope_2022)
past_erosion = np.where(clean_change < 0, np.abs(clean_change), 0)
past_erosion_risk = normalize_percentile(past_erosion)
geomorph_risk = np.clip(0.55*slope_score + 0.45*past_erosion_risk, 0, 1)

print("Crop shape:", d22.shape)
print("Difference correlation:", round(float(diff_correlation), 4))
print("Pixel size (m):", (xres, yres))


In [ ]:
def growth_form_stress(value):
    return {"crustose": 0.45, "foliose": 0.25, "fruticose": 0.15}.get(str(value).lower().strip(), 0.30)


def calculate_lichen_stress(row):
    coverage_stress = 1 - np.clip(row.coverage_percent / 80, 0, 1)
    stress = (0.35*row.chlorosis_score + 0.25*coverage_stress +
              0.25*row.nitrophyte_score + 0.15*growth_form_stress(row.growth_form))
    return float(np.clip(stress, 0, 1))


lichen_points = lichen_df.copy()
lichen_points["lichen_stress"] = lichen_points.apply(calculate_lichen_stress, axis=1)
xs, ys = warp_transform("EPSG:4326", profile_2022["crs"],
                        lichen_points.longitude.to_numpy(), lichen_points.latitude.to_numpy())
lichen_points["x"], lichen_points["y"] = xs, ys
pixel_pairs = [rasterio.transform.rowcol(transform_2022, x, y) for x, y in zip(xs, ys)]
lichen_points["row"] = [p[0] for p in pixel_pairs]
lichen_points["col"] = [p[1] for p in pixel_pairs]
height, width = d22.shape
lichen_points["inside_crop"] = (lichen_points.row.between(0, height-1) & lichen_points.col.between(0, width-1))
inside_points = lichen_points[lichen_points.inside_crop].copy()
if inside_points.empty:
    raise ValueError("No lichen observations fall inside the DEM crop.")

stress_grid = np.zeros(d22.shape, dtype='float32')
confidence_grid = np.zeros(d22.shape, dtype='float32')
presence_grid = np.zeros(d22.shape, dtype='float32')
for p in inside_points.itertuples():
    r, c = int(p.row), int(p.col)
    stress_grid[r, c] += float(p.lichen_stress) * float(p.cv_confidence)
    confidence_grid[r, c] += float(p.cv_confidence)
    presence_grid[r, c] += 1

smooth_stress = gaussian_filter(stress_grid, sigma=CONFIG.lichen_smoothing_sigma_px)
smooth_weight = gaussian_filter(confidence_grid, sigma=CONFIG.lichen_smoothing_sigma_px)
lichen_stress_surface = np.divide(smooth_stress, smooth_weight,
                                  out=np.zeros_like(smooth_stress), where=smooth_weight > 1e-12)
lichen_stress_surface = np.clip(lichen_stress_surface, 0, 1)

smooth_confidence = gaussian_filter(confidence_grid, sigma=CONFIG.lichen_smoothing_sigma_px)
smooth_presence = gaussian_filter(presence_grid, sigma=CONFIG.lichen_smoothing_sigma_px)
local_cv_confidence = np.divide(smooth_confidence, smooth_presence,
                                out=np.zeros_like(smooth_confidence), where=smooth_presence > 1e-12)
local_cv_confidence = np.clip(local_cv_confidence, 0, 1)

ecosignal_risk = np.clip(0.70*geomorph_risk + 0.30*lichen_stress_surface, 0, 1)
print(f"Mapped {len(inside_points)} real lichen observations into the aligned raster.")


## 4. Build real candidate sites and priority signals

Candidate points come directly from valid pixels of the real aligned raster. Terrain strata are derived from topographic position because this notebook does not yet contain a land-use/public-access layer. Consequently, every result is labeled `field_verification_required` rather than being presented as automatically accessible.


In [ ]:
# Topographic strata from the real 2022 DEM
elevation_norm = normalize_percentile(d22)
local_window_px = max(3, int(round(101 / np.mean([xres, yres]))))
if local_window_px % 2 == 0:
    local_window_px += 1
local_mean = uniform_filter(np.nan_to_num(d22, nan=float(np.nanmean(d22))), size=local_window_px)
tpi = d22 - local_mean
ridge_score = normalize_percentile(tpi)
valley_score = normalize_percentile(-tpi)

# Real-data uncertainty heuristic
sample_mask = np.zeros(d22.shape, dtype=bool)
sample_mask[inside_points.row.astype(int), inside_points.col.astype(int)] = True
distance_to_sample_m = distance_transform_edt(~sample_mask, sampling=(yres, xres))
distance_uncertainty = np.clip(distance_to_sample_m / CONFIG.distance_uncertainty_cap_m, 0, 1)
signal_disagreement = np.abs(geomorph_risk - lichen_stress_surface)
low_cv_confidence = 1 - local_cv_confidence
uncertainty_score = np.clip(0.45*distance_uncertainty + 0.35*signal_disagreement + 0.20*low_cv_confidence, 0, 1)

recent_change_score = normalize_percentile(np.abs(clean_change), low=50, high=98)
sparsity_score = distance_uncertainty.copy()
control_score = np.clip((0.35-ecosignal_risk)/0.35, 0, 1) * sparsity_score

candidate_step = max(1, int(round(CONFIG.candidate_spacing_m / np.mean([xres, yres]))))
rows = np.arange(0, height, candidate_step)
cols = np.arange(0, width, candidate_step)
rr, cc = np.meshgrid(rows, cols, indexing='ij')
rr, cc = rr.ravel(), cc.ravel()
valid_candidate = (np.isfinite(d22[rr, cc]) &
                   (distance_to_sample_m[rr, cc] >= CONFIG.exclude_near_sample_m))
rr, cc = rr[valid_candidate], cc[valid_candidate]

candidate_x, candidate_y = rasterio.transform.xy(transform_2022, rr, cc, offset='center')
candidate_lon, candidate_lat = warp_transform(profile_2022["crs"], "EPSG:4326", candidate_x, candidate_y)

candidates = pd.DataFrame({
    "row": rr, "col": cc, "x": candidate_x, "y": candidate_y,
    "longitude": candidate_lon, "latitude": candidate_lat,
    "risk_score": ecosignal_risk[rr, cc],
    "uncertainty_score": uncertainty_score[rr, cc],
    "sparsity_score": sparsity_score[rr, cc],
    "recent_change_score": recent_change_score[rr, cc],
    "control_score": control_score[rr, cc],
    "distance_to_nearest_sample_m": distance_to_sample_m[rr, cc],
    "slope_score": slope_score[rr, cc], "ridge_score": ridge_score[rr, cc],
    "valley_score": valley_score[rr, cc],
})

candidates["terrain_type"] = np.select(
    [candidates.valley_score >= 0.65,
     candidates.ridge_score >= 0.60,
     candidates.slope_score >= 0.45],
    ["lowland", "ridge", "slope"], default="other"
)

# Compare candidate terrain classes with the terrain classes already represented by observations.
sample_terrain = np.select(
    [valley_score[inside_points.row.astype(int), inside_points.col.astype(int)] >= 0.65,
     ridge_score[inside_points.row.astype(int), inside_points.col.astype(int)] >= 0.60,
     slope_score[inside_points.row.astype(int), inside_points.col.astype(int)] >= 0.45],
    ["lowland", "ridge", "slope"], default="other"
)
terrain_frequency = pd.Series(sample_terrain).value_counts(normalize=True)
candidates["terrain_diversity_score"] = 1 - candidates.terrain_type.map(terrain_frequency).fillna(0)

candidates["priority_score"] = 100 * (
    CONFIG.risk_weight*candidates.risk_score +
    CONFIG.uncertainty_weight*candidates.uncertainty_score +
    CONFIG.sparsity_weight*candidates.sparsity_score +
    CONFIG.recent_change_weight*candidates.recent_change_score +
    CONFIG.control_weight*candidates.control_score +
    CONFIG.terrain_diversity_weight*candidates.terrain_diversity_score
)

candidates["objective"] = np.select(
    [(candidates.control_score >= 0.45) & (candidates.risk_score <= 0.35),
     candidates.risk_score >= 0.65,
     candidates.uncertainty_score >= 0.65,
     candidates.recent_change_score >= 0.65],
    ["control", "hotspot_validation", "uncertainty_reduction", "change_validation"],
    default="coverage_gap"
)
candidates["access_status"] = "field_verification_required"
candidates.insert(0, "candidate_id", [f"ECOS-{i:05d}" for i in range(1, len(candidates)+1)])

print(f"Generated {len(candidates):,} candidates from the real raster grid.")
display(candidates.head())


## 4B. Real land-use and accessibility validation

This layer retrieves current OpenStreetMap features for the study bounds, labels real site types, and excludes explicit buildings, water/wetlands, industrial land, restricted areas, unsafe road offsets, major-road buffers, and railway buffers. Missing access tags remain `field_verification_required`; they are never treated as proof of public access.


In [ ]:
# ACTIVE SAMPLING PLANNER — VALIDATION STEP 1
# Paste this as a NEW CELL immediately after Section 4's candidate-building cell.

import importlib.util
import subprocess
import sys

if importlib.util.find_spec("osmnx") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "osmnx==2.1.0"
    ])

import geopandas as gpd
import osmnx as ox
from shapely.geometry import Point

ox.settings.use_cache = True
ox.settings.requests_timeout = 180

# Safety/access distances. These are conservative field-planning defaults,
# not legal determinations of where sampling is permitted.
ROADSIDE_MAX_DISTANCE_M = 40.0
MINIMUM_ROAD_OFFSET_M = 8.0
MAJOR_ROAD_EXCLUSION_M = 30.0

# OSMnx 2.1 expects (left, bottom, right, top) in EPSG:4326.
osm_bbox = tuple(float(v) for v in crop_bounds_wgs84)

# The query is a union: it retrieves every feature matching at least one tag.
osm_tags = {
    "highway": True,
    "building": True,
    "landuse": [
        "forest", "recreation_ground", "grass", "meadow", "education",
        "residential", "industrial", "commercial", "retail"
    ],
    "leisure": [
        "park", "nature_reserve", "garden", "recreation_ground",
        "playground", "sports_centre", "pitch"
    ],
    "amenity": ["school", "kindergarten", "college", "university"],
    "natural": ["wood", "water", "wetland"],
    "waterway": ["riverbank", "dock"],
    "railway": True,
}

try:
    osm_features = ox.features.features_from_bbox(osm_bbox, osm_tags)
except Exception as exc:
    raise RuntimeError(
        "OpenStreetMap retrieval failed. Wait 1–2 minutes and rerun this cell. "
        "If it fails again, confirm that the Colab runtime has internet access."
    ) from exc

if osm_features.empty:
    raise RuntimeError("OpenStreetMap returned no features for the EcoSignal study bounds.")

osm_projected = osm_features.to_crs(profile_2022["crs"]).copy()
osm_projected = osm_projected[osm_projected.geometry.notna() & ~osm_projected.geometry.is_empty]


def tag_matches(frame, column, allowed_values=None):
    """Safely match scalar or list-valued OSM tags."""
    if column not in frame.columns:
        return pd.Series(False, index=frame.index)

    allowed = None if allowed_values is None else {str(v).lower() for v in allowed_values}

    def match(value):
        if value is None or (isinstance(value, float) and np.isnan(value)):
            return False
        values = value if isinstance(value, (list, tuple, set, np.ndarray)) else [value]
        values = {str(v).lower() for v in values}
        return bool(values) if allowed is None else bool(values & allowed)

    return frame[column].apply(match)


def polygon_union(mask):
    polygon_types = osm_projected.geometry.geom_type.isin(["Polygon", "MultiPolygon"])
    geometry = osm_projected.loc[mask & polygon_types, "geometry"]
    return None if geometry.empty else geometry.union_all()


def linear_union(mask):
    line_types = osm_projected.geometry.geom_type.isin(["LineString", "MultiLineString"])
    geometry = osm_projected.loc[mask & line_types, "geometry"]
    return None if geometry.empty else geometry.union_all()


# Build real land-use and exclusion geometries.
road_mask = tag_matches(osm_projected, "highway")
major_road_mask = tag_matches(
    osm_projected,
    "highway",
    ["motorway", "motorway_link", "trunk", "trunk_link", "primary", "primary_link"]
)
forest_mask = (
    tag_matches(osm_projected, "landuse", ["forest"]) |
    tag_matches(osm_projected, "natural", ["wood"])
)
park_mask = (
    tag_matches(osm_projected, "leisure", ["park", "nature_reserve", "garden", "recreation_ground"]) |
    tag_matches(osm_projected, "landuse", ["recreation_ground"])
)
school_mask = (
    tag_matches(osm_projected, "amenity", ["school", "kindergarten", "college", "university"]) |
    tag_matches(osm_projected, "landuse", ["education"])
)
building_mask = tag_matches(osm_projected, "building")
water_mask = (
    tag_matches(osm_projected, "natural", ["water", "wetland"]) |
    tag_matches(osm_projected, "waterway", ["riverbank", "dock"])
)
industrial_mask = tag_matches(osm_projected, "landuse", ["industrial"])
railway_mask = tag_matches(osm_projected, "railway")
restricted_mask = (
    tag_matches(osm_projected, "access", ["no", "private"]) |
    tag_matches(osm_projected, "foot", ["no", "private"])
)

road_geometry = linear_union(road_mask)
major_road_geometry = linear_union(major_road_mask)
forest_geometry = polygon_union(forest_mask)
park_geometry = polygon_union(park_mask)
school_geometry = polygon_union(school_mask)
building_geometry = polygon_union(building_mask)
water_geometry = polygon_union(water_mask)
industrial_geometry = polygon_union(industrial_mask)
restricted_geometry = polygon_union(restricted_mask)
railway_geometry = linear_union(railway_mask)


def point_flags(points_gdf):
    """Add real OSM land-use, road-distance, and access flags to point features."""
    out = points_gdf.copy()

    def within(geometry):
        return pd.Series(False, index=out.index) if geometry is None else out.geometry.within(geometry)

    def distance(geometry):
        return pd.Series(np.inf, index=out.index) if geometry is None else out.geometry.distance(geometry)

    out["inside_forest"] = within(forest_geometry)
    out["inside_park"] = within(park_geometry)
    out["inside_school"] = within(school_geometry)
    out["inside_building"] = within(building_geometry)
    out["inside_water_or_wetland"] = within(water_geometry)
    out["inside_industrial"] = within(industrial_geometry)
    out["explicitly_restricted"] = within(restricted_geometry)
    out["distance_to_road_m"] = distance(road_geometry)
    out["distance_to_major_road_m"] = distance(major_road_geometry)
    out["distance_to_railway_m"] = distance(railway_geometry)

    # Precedence prevents a school or park near a road from being mislabeled roadside.
    out["site_type"] = np.select(
        [
            out.inside_school,
            out.inside_park,
            out.inside_forest,
            out.distance_to_road_m.between(MINIMUM_ROAD_OFFSET_M, ROADSIDE_MAX_DISTANCE_M),
            out.terrain_type.eq("slope"),
            out.terrain_type.eq("lowland"),
            out.terrain_type.eq("ridge"),
        ],
        ["school", "park", "forest", "roadside", "slope", "lowland", "ridge"],
        default="other",
    )

    # OSM absence is not evidence of public access.
    out["access_status"] = np.select(
        [
            out.explicitly_restricted,
            out.inside_school,
            out.inside_park,
        ],
        ["excluded_private_or_no_access", "permission_required", "likely_public_verify_hours"],
        default="field_verification_required",
    )

    out["exclusion_reason"] = np.select(
        [
            out.inside_building,
            out.inside_water_or_wetland,
            out.inside_industrial,
            out.explicitly_restricted,
            out.distance_to_major_road_m < MAJOR_ROAD_EXCLUSION_M,
            out.distance_to_road_m < MINIMUM_ROAD_OFFSET_M,
            out.distance_to_railway_m < MINIMUM_ROAD_OFFSET_M,
        ],
        [
            "building_footprint",
            "water_or_wetland",
            "industrial_land",
            "explicit_private_or_no_access",
            "too_close_to_major_road",
            "too_close_to_road_centerline",
            "too_close_to_railway",
        ],
        default="",
    )
    out["eligible_for_sampling"] = out.exclusion_reason.eq("")
    return out


# Enrich and filter candidate sites.
candidate_points = gpd.GeoDataFrame(
    candidates.copy(),
    geometry=gpd.points_from_xy(candidates.x, candidates.y),
    crs=profile_2022["crs"],
)
candidate_points = point_flags(candidate_points)

# Apply the same site-type classifier to existing observations so diversity
# rewards site types that are genuinely underrepresented in the real sample set.
sample_base = inside_points.copy()
sample_base["terrain_type"] = np.select(
    [
        valley_score[sample_base.row.astype(int), sample_base.col.astype(int)] >= 0.65,
        ridge_score[sample_base.row.astype(int), sample_base.col.astype(int)] >= 0.60,
        slope_score[sample_base.row.astype(int), sample_base.col.astype(int)] >= 0.45,
    ],
    ["lowland", "ridge", "slope"],
    default="other",
)
sample_points = gpd.GeoDataFrame(
    sample_base,
    geometry=gpd.points_from_xy(sample_base.x, sample_base.y),
    crs=profile_2022["crs"],
)
sample_points = point_flags(sample_points)

site_frequency = sample_points.site_type.value_counts(normalize=True)
candidate_points["terrain_diversity_score"] = (
    1 - candidate_points.site_type.map(site_frequency).fillna(0)
).clip(0, 1)

# Recalculate the original planner priority after replacing terrain-only
# diversity with real land-use/site-type diversity.
candidate_points["priority_score"] = 100 * (
    CONFIG.risk_weight * candidate_points.risk_score +
    CONFIG.uncertainty_weight * candidate_points.uncertainty_score +
    CONFIG.sparsity_weight * candidate_points.sparsity_score +
    CONFIG.recent_change_weight * candidate_points.recent_change_score +
    CONFIG.control_weight * candidate_points.control_score +
    CONFIG.terrain_diversity_weight * candidate_points.terrain_diversity_score
)

before_count = len(candidate_points)
exclusion_summary = (
    candidate_points.loc[~candidate_points.eligible_for_sampling, "exclusion_reason"]
    .value_counts()
    .to_dict()
)

# Replace `candidates` so the existing Section 5 selection code uses the
# land-use-enriched, access-screened candidate table automatically.
candidates = candidate_points[candidate_points.eligible_for_sampling].copy()
candidates = candidates.drop(columns="geometry")

print("OSM features retrieved:", len(osm_projected))
print("Candidates before access/safety screening:", before_count)
print("Candidates retained:", len(candidates))
print("Candidates excluded:", before_count - len(candidates))
print("Exclusion summary:", exclusion_summary)
print("\nRetained candidate site types:")
display(candidates.site_type.value_counts().rename("candidate_count").to_frame())
print("\nAccess-status labels:")
display(candidates.access_status.value_counts().rename("candidate_count").to_frame())

# Validation checks on real OSM-enriched results.
assert len(candidates) > 0, "All candidates were excluded; inspect the OSM overlays."
assert not candidates.inside_building.any()
assert not candidates.inside_water_or_wetland.any()
assert not candidates.inside_industrial.any()
assert not candidates.explicitly_restricted.any()
assert (candidates.distance_to_major_road_m >= MAJOR_ROAD_EXCLUSION_M).all()
assert (candidates.distance_to_road_m >= MINIMUM_ROAD_OFFSET_M).all()
assert candidates.site_type.notna().all()
assert candidates.access_status.notna().all()

print("\nPASS: Land-use/accessibility validation completed.")
print("Now rerun Sections 5–8 to regenerate recommendations, the map, QA, and exports.")


## 5. Select a scientifically balanced field plan

The selector reserves space for controls, hotspots, and uncertainty checks when qualifying candidates exist, then fills remaining positions by overall priority. All selected sites must satisfy the minimum geographic spacing.


In [ ]:
# ACTIVE SAMPLING PLANNER — REVISED SECTION 5
# Replace the current Section 5 code cell with this entire cell.
# Prerequisite: run Sections 1–4 and the OSM validation cell first.

EDGE_BUFFER_M = 75.0
MAX_SCHOOL_SITES = 4
MAX_ROADSIDE_SITES = 3
MAX_OTHER_SINGLE_TYPE = 3
MAX_PERMISSION_REQUIRED = 4
TARGET_DISTINCT_SITE_TYPES = 4

# ------------------------------------------------------------------
# 1. Remove crop-edge artifacts
# ------------------------------------------------------------------
# Distance-based uncertainty naturally rises toward an arbitrary raster edge.
# These pixels are excluded unless a future study-area polygon defines a real edge.
candidates = candidates.copy()
candidates["distance_to_crop_edge_m"] = np.minimum.reduce([
    candidates["row"].to_numpy(float) * yres,
    (height - 1 - candidates["row"].to_numpy(float)) * yres,
    candidates["col"].to_numpy(float) * xres,
    (width - 1 - candidates["col"].to_numpy(float)) * xres,
])

edge_excluded = int((candidates.distance_to_crop_edge_m < EDGE_BUFFER_M).sum())
candidates = candidates[candidates.distance_to_crop_edge_m >= EDGE_BUFFER_M].copy()

# ------------------------------------------------------------------
# 2. Redefine a valid scientific control
# ------------------------------------------------------------------
# A control must be low risk, sparsely sampled, AND relatively stable.
# This prevents strong landscape-change zones from being mislabeled controls.
low_risk_suitability = np.clip((0.30 - candidates.risk_score) / 0.30, 0, 1)
landscape_stability = np.clip(1 - candidates.recent_change_score, 0, 1)

candidates["control_score"] = (
    low_risk_suitability *
    candidates.sparsity_score *
    landscape_stability
)

candidates["strict_control_eligible"] = (
    (candidates.risk_score <= 0.30) &
    (candidates.recent_change_score <= 0.35) &
    (candidates.control_score >= 0.25)
)

# Recalculate priority with the corrected control score.
candidates["priority_score"] = 100 * (
    CONFIG.risk_weight * candidates.risk_score +
    CONFIG.uncertainty_weight * candidates.uncertainty_score +
    CONFIG.sparsity_weight * candidates.sparsity_score +
    CONFIG.recent_change_weight * candidates.recent_change_score +
    CONFIG.control_weight * candidates.control_score +
    CONFIG.terrain_diversity_weight * candidates.terrain_diversity_score
)

# Assign one primary field objective. Control eligibility receives precedence
# only when all strict control requirements are met.
candidates["objective"] = np.select(
    [
        candidates.strict_control_eligible,
        candidates.risk_score >= 0.65,
        candidates.uncertainty_score >= 0.65,
        candidates.recent_change_score >= 0.65,
    ],
    ["control", "hotspot_validation", "uncertainty_reduction", "change_validation"],
    default="coverage_gap",
)

# ------------------------------------------------------------------
# 3. Explanation logic
# ------------------------------------------------------------------
CONTRIBUTION_LABELS = {
    "risk": "elevated EcoSignal risk",
    "uncertainty": "high model uncertainty",
    "sparsity": "low existing sample density",
    "recent_change": "strong recent elevation change",
    "control": "suitability as a stable low-risk control",
    "terrain_diversity": "underrepresented site type",
}


def add_explanation(row):
    values = {
        "risk": CONFIG.risk_weight * row.risk_score,
        "uncertainty": CONFIG.uncertainty_weight * row.uncertainty_score,
        "sparsity": CONFIG.sparsity_weight * row.sparsity_score,
        "recent_change": CONFIG.recent_change_weight * row.recent_change_score,
        "control": CONFIG.control_weight * row.control_score,
        "terrain_diversity": CONFIG.terrain_diversity_weight * row.terrain_diversity_score,
    }
    top = sorted(values, key=values.get, reverse=True)[:3]
    reason = "; ".join(CONTRIBUTION_LABELS[key] for key in top).capitalize() + "."
    actions = {
        "control": "Collect a stable low-exposure control sample.",
        "hotspot_validation": "Collect a targeted sample to validate this predicted hotspot.",
        "uncertainty_reduction": "Collect a verification sample to reduce model uncertainty.",
        "change_validation": "Collect a sample to validate the recent-change signal.",
        "coverage_gap": "Collect a sample to improve spatial and site-type coverage.",
    }
    return actions[row.objective], reason


# ------------------------------------------------------------------
# 4. Balanced, spatially separated selection
# ------------------------------------------------------------------
def far_enough(row, selected, minimum_m):
    if not selected:
        return True
    return min(math.hypot(row.x - chosen.x, row.y - chosen.y) for chosen in selected) >= minimum_m


def within_caps(row, selected, relaxed=False):
    site_counts = pd.Series([chosen.site_type for chosen in selected]).value_counts()
    permission_count = sum(chosen.access_status == "permission_required" for chosen in selected)

    if relaxed:
        type_cap = 5
        permission_cap = 5
    else:
        type_cap = {
            "school": MAX_SCHOOL_SITES,
            "roadside": MAX_ROADSIDE_SITES,
        }.get(row.site_type, MAX_OTHER_SINGLE_TYPE)
        permission_cap = MAX_PERMISSION_REQUIRED

    if site_counts.get(row.site_type, 0) >= type_cap:
        return False
    if row.access_status == "permission_required" and permission_count >= permission_cap:
        return False
    return True


def choose_from(pool, count, selected, relaxed=False):
    chosen_ids = {row.candidate_id for row in selected}
    for row in pool.sort_values("priority_score", ascending=False).itertuples():
        if count <= 0 or len(selected) >= CONFIG.recommendations:
            break
        if row.candidate_id in chosen_ids:
            continue
        if not far_enough(row, selected, CONFIG.minimum_selected_spacing_m):
            continue
        if not within_caps(row, selected, relaxed=relaxed):
            continue
        selected.append(row)
        chosen_ids.add(row.candidate_id)
        count -= 1
    return selected


selected_rows = []

# Exact scientific targets rather than unlimited minimum quotas.
selection_targets = [
    ("control", 2),
    ("hotspot_validation", 3),
    ("uncertainty_reduction", 2),
]
for objective, target in selection_targets:
    selected_rows = choose_from(
        candidates[candidates.objective == objective],
        target,
        selected_rows,
    )

# Add underrepresented real site types until the plan has adequate diversity.
available_types = candidates.site_type.value_counts().index.tolist()
type_order = sorted(
    available_types,
    key=lambda site: (site_frequency.get(site, 0), -int((candidates.site_type == site).sum())),
)
for site_type in type_order:
    selected_types = {row.site_type for row in selected_rows}
    if len(selected_types) >= min(TARGET_DISTINCT_SITE_TYPES, len(available_types)):
        break
    if site_type not in selected_types:
        selected_rows = choose_from(
            candidates[candidates.site_type == site_type],
            1,
            selected_rows,
        )

# Fill remaining positions by total priority while preserving caps.
selected_rows = choose_from(
    candidates,
    CONFIG.recommendations - len(selected_rows),
    selected_rows,
)

# If strict caps plus spacing prevent ten selections, relax caps slightly but
# never relax safety/access exclusions or geographic spacing.
if len(selected_rows) < CONFIG.recommendations:
    selected_rows = choose_from(
        candidates,
        CONFIG.recommendations - len(selected_rows),
        selected_rows,
        relaxed=True,
    )

recommendations = pd.DataFrame([row._asdict() for row in selected_rows])
if recommendations.empty:
    raise RuntimeError("No candidate sites could be selected after validation.")

recommendations = recommendations.sort_values("priority_score", ascending=False).reset_index(drop=True)
recommendations.insert(0, "rank", np.arange(1, len(recommendations) + 1))
explanations = recommendations.apply(add_explanation, axis=1, result_type="expand")
recommendations["recommended_action"] = explanations[0]
recommendations["reason"] = explanations[1]

# ------------------------------------------------------------------
# 5. Validation and display
# ------------------------------------------------------------------
controls = recommendations[recommendations.objective == "control"]
assert (controls.risk_score <= 0.30).all(), "A control has excessive risk."
assert (controls.recent_change_score <= 0.35).all(), "A control is not landscape-stable."
assert (recommendations.distance_to_crop_edge_m >= EDGE_BUFFER_M).all()
assert recommendations.site_type.nunique() >= min(
    TARGET_DISTINCT_SITE_TYPES,
    candidates.site_type.nunique(),
)
assert (recommendations.access_status == "permission_required").sum() <= 5

print("Crop-edge candidates excluded:", edge_excluded)
print("Strict stable-control candidates available:", int(candidates.strict_control_eligible.sum()))
if candidates.strict_control_eligible.sum() < 2:
    print(
        "WARNING: Fewer than two scientifically valid controls exist inside this study area. "
        "The planner will not invent invalid controls; consider expanding the crop boundary."
    )

show_columns = [
    "rank", "candidate_id", "objective", "site_type", "terrain_type",
    "priority_score", "risk_score", "uncertainty_score", "recent_change_score",
    "distance_to_nearest_sample_m", "distance_to_crop_edge_m",
    "latitude", "longitude", "recommended_action", "reason", "access_status",
]
display(recommendations[show_columns].round(4))

print("\nSelected objectives:")
display(recommendations.objective.value_counts().rename("selected_count").to_frame())
print("\nSelected site types:")
display(recommendations.site_type.value_counts().rename("selected_count").to_frame())
print("\nPASS: Stable controls, edge protection, access screening, spacing, and site diversity enforced.")
print("Now rerun Sections 6–8 and send the printed summaries plus the revised map.")


## 6. Visual review on the real risk map

Blue stars are the recommended next samples. Existing observations are white × markers. Candidate colors come from the real aligned EcoSignal risk raster.


In [ ]:
left, bottom, right, top = rasterio.transform.array_bounds(height, width, transform_2022)
fig, ax = plt.subplots(figsize=(12, 9))
im = ax.imshow(ecosignal_risk, cmap='inferno', vmin=0, vmax=1,
               extent=[left, right, bottom, top], origin='upper')
ax.scatter(inside_points.x, inside_points.y, marker='x', s=65, color='white',
           linewidth=1.5, label='Existing lichen samples')
ax.scatter(recommendations.x, recommendations.y, marker='*', s=250,
           color='#27a7ff', edgecolor='white', linewidth=1.0, label='Recommended next samples')
for r in recommendations.itertuples():
    ax.annotate(str(r.rank), (r.x, r.y), xytext=(7, 7), textcoords='offset points',
                color='white', weight='bold', fontsize=10)
plt.colorbar(im, ax=ax, label='EcoSignal risk')
ax.set(title='EcoSignal Active Sampling Plan — Real GPS-Aligned Data',
       xlabel=f'X ({profile_2022["crs"]})', ylabel=f'Y ({profile_2022["crs"]})')
ax.legend(loc='best')
plt.show()


## 7. Scientific QA

These tests run on the real outputs. They check bounds, coordinate validity, uniqueness, score ranges, exclusion around existing samples, and spacing between selected sites.


In [ ]:
def run_qa():
    assert len(recommendations) <= CONFIG.recommendations
    assert recommendations.candidate_id.is_unique
    assert recommendations.priority_score.between(0, 100).all()
    assert recommendations.risk_score.between(0, 1).all()
    assert recommendations.uncertainty_score.between(0, 1).all()
    assert recommendations.latitude.between(-90, 90).all()
    assert recommendations.longitude.between(-180, 180).all()
    assert (recommendations.distance_to_nearest_sample_m >= CONFIG.exclude_near_sample_m-1e-6).all()
    minimum_pair_spacing = None
    if len(recommendations) > 1:
        xy = recommendations[["x", "y"]].to_numpy(float)
        delta = xy[:, None, :] - xy[None, :, :]
        distances = np.sqrt((delta**2).sum(axis=2))
        np.fill_diagonal(distances, np.inf)
        minimum_pair_spacing = float(distances.min())
        assert minimum_pair_spacing >= CONFIG.minimum_selected_spacing_m-1e-6
    assert recommendations.reason.str.len().gt(20).all()
    return {
        "status": "PASS", "real_lichen_observations": int(len(inside_points)),
        "candidate_sites": int(len(candidates)), "selected_sites": int(len(recommendations)),
        "minimum_pair_spacing_m": None if minimum_pair_spacing is None else round(minimum_pair_spacing, 2),
        "difference_correlation": round(float(diff_correlation), 4),
        "objectives_selected": recommendations.objective.value_counts().to_dict(),
    }

qa_results = run_qa()
print(json.dumps(qa_results, indent=2))


## 8. Export for EcoSignal

The CSV is analysis-friendly. The GeoJSON is frontend/map-ready and includes the explanation, objective, scores, and the warning that access still requires field verification.


In [ ]:
OUTPUT_DIR = DEM_DIR / "ActiveSamplingPlanner"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

csv_path = OUTPUT_DIR / "active_sampling_recommendations.csv"
geojson_path = OUTPUT_DIR / "active_sampling_recommendations.geojson"
qa_path = OUTPUT_DIR / "active_sampling_qa.json"
recommendations.to_csv(csv_path, index=False)

properties_to_export = [
    "rank", "candidate_id", "objective", "site_type", "terrain_type",
    "priority_score", "risk_score", "uncertainty_score", "sparsity_score",
    "recent_change_score", "control_score", "terrain_diversity_score",
    "distance_to_nearest_sample_m", "distance_to_crop_edge_m",
    "recommended_action", "reason", "access_status"
]
features = []
for row in recommendations.itertuples():
    props = {}
    for key in properties_to_export:
        value = getattr(row, key)
        if isinstance(value, (np.integer, np.floating)):
            value = value.item()
        props[key] = value
    features.append({
        "type": "Feature",
        "geometry": {"type": "Point", "coordinates": [float(row.longitude), float(row.latitude)]},
        "properties": props,
    })

geojson = {
    "type": "FeatureCollection",
    "name": "EcoSignal Active Sampling Recommendations",
    "planner_version": "2.0.0",
    "uncertainty_type": "heuristic_not_calibrated",
    "features": features,
}
geojson_path.write_text(json.dumps(geojson, indent=2))
qa_path.write_text(json.dumps(qa_results, indent=2))

print("Saved validated planner outputs:")
print("-", csv_path)
print("-", geojson_path)
print("-", qa_path)


## 9. Weight-sensitivity validation

This test perturbs all six planner weights by up to ±20% across 250 simulations, reruns the full balanced and spatially constrained selection process, and measures whether the final field plan remains stable. It does not change the production recommendations.


In [ ]:
SENSITIVITY_RUNS = 250
SENSITIVITY_SEED = 20260711
WEIGHT_NAMES = ["risk", "uncertainty", "sparsity", "recent_change", "control", "site_diversity"]
base_weights = np.array([
    CONFIG.risk_weight,
    CONFIG.uncertainty_weight,
    CONFIG.sparsity_weight,
    CONFIG.recent_change_weight,
    CONFIG.control_weight,
    CONFIG.terrain_diversity_weight,
], dtype=float)


def score_with_weights(frame, weights):
    values = np.column_stack([
        frame.risk_score,
        frame.uncertainty_score,
        frame.sparsity_score,
        frame.recent_change_score,
        frame.control_score,
        frame.terrain_diversity_score,
    ])
    return 100 * (values @ weights)


def select_plan_for_weights(weights):
    scenario = candidates.copy()
    scenario["priority_score"] = score_with_weights(scenario, weights)
    selected = []

    for objective, target in [
        ("control", 2),
        ("hotspot_validation", 3),
        ("uncertainty_reduction", 2),
    ]:
        selected = choose_from(scenario[scenario.objective == objective], target, selected)

    available = scenario.site_type.value_counts().index.tolist()
    ordered_types = sorted(
        available,
        key=lambda site: (site_frequency.get(site, 0), -int((scenario.site_type == site).sum())),
    )
    for site_type in ordered_types:
        if len({row.site_type for row in selected}) >= min(TARGET_DISTINCT_SITE_TYPES, len(available)):
            break
        if site_type not in {row.site_type for row in selected}:
            selected = choose_from(scenario[scenario.site_type == site_type], 1, selected)

    selected = choose_from(scenario, CONFIG.recommendations-len(selected), selected)
    if len(selected) < CONFIG.recommendations:
        selected = choose_from(
            scenario,
            CONFIG.recommendations-len(selected),
            selected,
            relaxed=True,
        )
    return [row.candidate_id for row in selected]


rng = np.random.default_rng(SENSITIVITY_SEED)
baseline_ids = set(recommendations.candidate_id)
selection_counts = {candidate_id: 0 for candidate_id in candidates.candidate_id}
simulation_rows = []

for run in range(SENSITIVITY_RUNS):
    multipliers = rng.uniform(0.80, 1.20, len(base_weights))
    weights = base_weights * multipliers
    weights = weights / weights.sum()
    selected_ids = set(select_plan_for_weights(weights))
    for candidate_id in selected_ids:
        selection_counts[candidate_id] += 1
    union = baseline_ids | selected_ids
    jaccard = len(baseline_ids & selected_ids) / len(union) if union else 1.0
    row = {"run": run+1, "jaccard_vs_baseline": jaccard}
    row.update({f"weight_{name}": float(value) for name, value in zip(WEIGHT_NAMES, weights)})
    simulation_rows.append(row)

sensitivity_runs = pd.DataFrame(simulation_rows)
selection_stability = candidates[["candidate_id", "objective", "site_type", "latitude", "longitude"]].copy()
selection_stability["selection_frequency"] = selection_stability.candidate_id.map(selection_counts) / SENSITIVITY_RUNS
selection_stability["in_baseline_plan"] = selection_stability.candidate_id.isin(baseline_ids)
selection_stability = selection_stability.sort_values("selection_frequency", ascending=False)

baseline_stability = selection_stability[selection_stability.in_baseline_plan].copy()
median_jaccard = float(sensitivity_runs.jaccard_vs_baseline.median())
minimum_jaccard = float(sensitivity_runs.jaccard_vs_baseline.min())
core_site_count = int((baseline_stability.selection_frequency >= 0.70).sum())

if median_jaccard >= 0.65:
    robustness_grade = "STRONG"
elif median_jaccard >= 0.45:
    robustness_grade = "ACCEPTABLE"
else:
    robustness_grade = "FRAGILE"

sensitivity_summary = {
    "runs": SENSITIVITY_RUNS,
    "perturbation": "independent uniform ±20%, renormalized",
    "median_jaccard_vs_baseline": round(median_jaccard, 4),
    "minimum_jaccard_vs_baseline": round(minimum_jaccard, 4),
    "baseline_sites_selected_at_least_70_percent": core_site_count,
    "robustness_grade": robustness_grade,
}

print(json.dumps(sensitivity_summary, indent=2))
display(baseline_stability[["candidate_id", "objective", "site_type", "selection_frequency"]].round(3))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].hist(sensitivity_runs.jaccard_vs_baseline, bins=np.linspace(0, 1, 21), color="#386cb0", edgecolor="white")
axes[0].axvline(median_jaccard, color="black", linestyle="--", label=f"Median={median_jaccard:.2f}")
axes[0].set(title="Plan stability under ±20% weight changes", xlabel="Jaccard similarity", ylabel="Simulations")
axes[0].legend()
ordered = baseline_stability.sort_values("selection_frequency")
axes[1].barh(ordered.candidate_id, ordered.selection_frequency, color="#7fc97f")
axes[1].axvline(0.70, color="black", linestyle="--", label="Core-site threshold")
axes[1].set(title="Baseline recommendation stability", xlabel="Selection frequency", xlim=(0, 1))
axes[1].legend()
plt.tight_layout()
plt.show()

sensitivity_runs.to_csv(OUTPUT_DIR/"weight_sensitivity_runs.csv", index=False)
selection_stability.to_csv(OUTPUT_DIR/"recommendation_stability.csv", index=False)
(OUTPUT_DIR/"weight_sensitivity_summary.json").write_text(json.dumps(sensitivity_summary, indent=2))


## 10. Confidence and Uncertainty Layer v1

The layer combines five transparent evidence components:

1. **Spatial extrapolation:** distance from real lichen observations.
2. **Signal disagreement:** difference between geomorphology and lichen-stress evidence.
3. **CV uncertainty:** confidence of the nearest real lichen observation.
4. **Risk-category ambiguity:** closeness to Low/Moderate/High classification boundaries.
5. **Boundary uncertainty:** proximity to the artificial raster crop edge.

The result is an **evidence-confidence score**, not a calibrated probability that the risk prediction is correct. Statistical calibration requires substantially more than the current four independent field observations.


In [ ]:
CONFIDENCE_COMPONENT_WEIGHTS = {
    "spatial_extrapolation": 0.35,
    "signal_disagreement": 0.25,
    "cv_uncertainty": 0.20,
    "category_ambiguity": 0.15,
    "boundary_uncertainty": 0.05,
}
if not np.isclose(sum(CONFIDENCE_COMPONENT_WEIGHTS.values()), 1.0):
    raise ValueError("Confidence component weights must total 1.0.")

valid_map = np.isfinite(d22) & np.isfinite(ecosignal_risk)

# 1. Spatial extrapolation uncertainty
spatial_uncertainty = np.clip(
    distance_to_sample_m / CONFIG.distance_uncertainty_cap_m,
    0,
    1,
)

# 2. Independent evidence disagreement
disagreement_uncertainty = np.clip(np.abs(geomorph_risk-lichen_stress_surface), 0, 1)

# 3. Nearest-observation CV uncertainty (distance remains a separate component)
sample_cv_grid = np.zeros(d22.shape, dtype="float32")
for point in inside_points.itertuples():
    r, c = int(point.row), int(point.col)
    sample_cv_grid[r, c] = max(sample_cv_grid[r, c], float(point.cv_confidence))

_, nearest_indices = distance_transform_edt(
    ~sample_mask,
    sampling=(yres, xres),
    return_indices=True,
)
nearest_cv_confidence = sample_cv_grid[nearest_indices[0], nearest_indices[1]]
cv_uncertainty = np.clip(1-nearest_cv_confidence, 0, 1)

# 4. Risk-category ambiguity around 0.33 and 0.67 thresholds
distance_to_risk_boundary = np.minimum(
    np.abs(ecosignal_risk-0.33),
    np.abs(ecosignal_risk-0.67),
)
category_uncertainty = 1-np.clip(distance_to_risk_boundary/0.165, 0, 1)

# 5. Artificial crop-boundary uncertainty
grid_rows, grid_cols = np.indices(d22.shape)
distance_to_edge_m = np.minimum.reduce([
    grid_rows*yres,
    (height-1-grid_rows)*yres,
    grid_cols*xres,
    (width-1-grid_cols)*xres,
])
boundary_uncertainty = 1-np.clip(distance_to_edge_m/150.0, 0, 1)

component_arrays = {
    "spatial_extrapolation": spatial_uncertainty,
    "signal_disagreement": disagreement_uncertainty,
    "cv_uncertainty": cv_uncertainty,
    "category_ambiguity": category_uncertainty,
    "boundary_uncertainty": boundary_uncertainty,
}

uncertainty_layer = sum(
    CONFIDENCE_COMPONENT_WEIGHTS[name]*array
    for name, array in component_arrays.items()
)
uncertainty_layer = np.where(valid_map, np.clip(uncertainty_layer, 0, 1), np.nan)
confidence_layer = np.where(valid_map, 100*(1-uncertainty_layer), np.nan).astype("float32")

risk_class = np.full(d22.shape, "No data", dtype=object)
risk_class[valid_map & (ecosignal_risk < 0.33)] = "Low"
risk_class[valid_map & (ecosignal_risk >= 0.33) & (ecosignal_risk < 0.67)] = "Moderate"
risk_class[valid_map & (ecosignal_risk >= 0.67)] = "High"

confidence_class = np.full(d22.shape, "No data", dtype=object)
confidence_class[valid_map & (confidence_layer >= 75)] = "High confidence"
confidence_class[valid_map & (confidence_layer >= 50) & (confidence_layer < 75)] = "Moderate confidence"
confidence_class[valid_map & (confidence_layer < 50)] = "Low confidence"

needs_more_samples = valid_map & (
    (uncertainty_layer >= 0.60) |
    (distance_to_sample_m >= 500) |
    (disagreement_uncertainty >= 0.45)
)

component_stack = np.stack([component_arrays[name] for name in component_arrays])
dominant_component_code = np.argmax(component_stack, axis=0).astype("uint8")
component_names = list(component_arrays)
reason_text = {
    0: "far from existing lichen observations",
    1: "geomorphology and lichen signals disagree",
    2: "nearest lichen CV result has limited confidence",
    3: "risk score is close to a category boundary",
    4: "location is close to the artificial map boundary",
}

print("Confidence layer calculated from real EcoSignal evidence.")


### Confidence maps and zone summary


In [ ]:
valid_confidence = confidence_layer[valid_map]
confidence_summary = {
    "method": "evidence_confidence_heuristic_v1_not_calibrated",
    "real_field_observations": int(len(inside_points)),
    "mean_confidence": round(float(np.nanmean(valid_confidence)), 2),
    "median_confidence": round(float(np.nanmedian(valid_confidence)), 2),
    "high_confidence_percent": round(float(100*np.mean(valid_confidence >= 75)), 2),
    "moderate_confidence_percent": round(float(100*np.mean((valid_confidence >= 50) & (valid_confidence < 75))), 2),
    "low_confidence_percent": round(float(100*np.mean(valid_confidence < 50)), 2),
    "needs_more_samples_percent": round(float(100*np.mean(needs_more_samples[valid_map])), 2),
    "component_weights": CONFIDENCE_COMPONENT_WEIGHTS,
}
print(json.dumps(confidence_summary, indent=2))

left, bottom, right, top = rasterio.transform.array_bounds(height, width, transform_2022)
fig, axes = plt.subplots(1, 3, figsize=(19, 6))

im0 = axes[0].imshow(ecosignal_risk, cmap="inferno", vmin=0, vmax=1,
                     extent=[left, right, bottom, top], origin="upper")
axes[0].set_title("Environmental risk")
plt.colorbar(im0, ax=axes[0], fraction=0.046, label="Risk score")

im1 = axes[1].imshow(confidence_layer, cmap="RdYlGn", vmin=0, vmax=100,
                     extent=[left, right, bottom, top], origin="upper")
axes[1].set_title("Evidence confidence")
plt.colorbar(im1, ax=axes[1], fraction=0.046, label="Confidence, 0–100")

axes[2].imshow(ecosignal_risk, cmap="gray", vmin=0, vmax=1,
               extent=[left, right, bottom, top], origin="upper", alpha=0.45)
axes[2].imshow(np.where(needs_more_samples, 1, np.nan), cmap="Reds", vmin=0, vmax=1,
               extent=[left, right, bottom, top], origin="upper", alpha=0.75)
axes[2].scatter(inside_points.x, inside_points.y, marker="x", color="#00ffff", s=65,
                linewidth=1.5, label="Existing samples")
axes[2].set_title("Needs more samples")
axes[2].legend(loc="best")

for ax in axes:
    ax.set_xlabel(f"X ({profile_2022['crs']})")
    ax.set_ylabel(f"Y ({profile_2022['crs']})")

plt.tight_layout()
plt.show()


### Per-sample and recommendation confidence

Each row reports risk, confidence, and the primary reason confidence is limited.


In [ ]:
def extract_confidence_table(frame, id_column):
    out = frame.copy()
    rows = out.row.astype(int).to_numpy()
    cols = out.col.astype(int).to_numpy()
    out["environmental_risk"] = ecosignal_risk[rows, cols]
    out["risk_class"] = risk_class[rows, cols]
    out["confidence_score"] = confidence_layer[rows, cols]
    out["confidence_class"] = confidence_class[rows, cols]
    out["uncertainty_score_v1"] = uncertainty_layer[rows, cols]
    out["uncertainty_reason"] = [reason_text[int(code)] for code in dominant_component_code[rows, cols]]
    out["needs_more_samples"] = needs_more_samples[rows, cols]
    columns = [id_column, "latitude", "longitude", "environmental_risk", "risk_class",
               "confidence_score", "confidence_class", "uncertainty_score_v1",
               "uncertainty_reason", "needs_more_samples"]
    return out[columns]

sample_source = inside_points.reset_index(drop=True).copy()
sample_source["sample_id"] = [f"LICHEN-{i:03d}" for i in range(1, len(sample_source)+1)]
sample_confidence = extract_confidence_table(sample_source, "sample_id")
recommendation_confidence = extract_confidence_table(recommendations, "candidate_id")

print("Existing real observations:")
display(sample_confidence.round(3))
print("Recommended next samples:")
display(recommendation_confidence.round(3))


### Export confidence rasters, zones, tables, and metadata


In [ ]:
from rasterio.features import shapes
from shapely.geometry import shape

CONFIDENCE_OUTPUT_DIR = DEM_DIR / "ConfidenceUncertaintyLayer"
CONFIDENCE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def write_float_raster(path, values, description):
    nodata = -9999.0
    profile = profile_2022.copy()
    profile.update(dtype="float32", count=1, nodata=nodata, compress="lzw")
    output = np.where(np.isfinite(values), values, nodata).astype("float32")
    with rasterio.open(path, "w", **profile) as dst:
        dst.write(output, 1)
        dst.set_band_description(1, description)


write_float_raster(CONFIDENCE_OUTPUT_DIR/"ecosignal_risk.tif", ecosignal_risk, "EcoSignal risk 0-1")
write_float_raster(CONFIDENCE_OUTPUT_DIR/"ecosignal_uncertainty_v1.tif", uncertainty_layer, "Evidence uncertainty 0-1")
write_float_raster(CONFIDENCE_OUTPUT_DIR/"ecosignal_confidence_v1.tif", confidence_layer, "Evidence confidence 0-100")

reason_profile = profile_2022.copy()
reason_profile.update(dtype="uint8", count=1, nodata=255, compress="lzw")
reason_output = np.where(valid_map, dominant_component_code, 255).astype("uint8")
with rasterio.open(CONFIDENCE_OUTPUT_DIR/"uncertainty_reason_code.tif", "w", **reason_profile) as dst:
    dst.write(reason_output, 1)
    dst.set_band_description(1, "Dominant uncertainty reason code")

# Convert contiguous needs-more-samples zones to polygons and remove tiny noise.
zone_records = []
for geometry, value in shapes(
    needs_more_samples.astype("uint8"),
    mask=valid_map,
    transform=transform_2022,
):
    if int(value) != 1:
        continue
    polygon = shape(geometry)
    if polygon.area < 2_500:
        continue
    zone_records.append({"geometry": polygon, "area_m2": float(polygon.area)})

if zone_records:
    zones = gpd.GeoDataFrame(zone_records, crs=profile_2022["crs"]).to_crs("EPSG:4326")
    zones.insert(0, "zone_id", [f"UNCERTAINTY-{i:03d}" for i in range(1, len(zones)+1)])
    zones.to_file(CONFIDENCE_OUTPUT_DIR/"needs_more_samples_zones.geojson", driver="GeoJSON")
else:
    zones = gpd.GeoDataFrame(columns=["zone_id", "area_m2", "geometry"], geometry="geometry", crs="EPSG:4326")
    (CONFIDENCE_OUTPUT_DIR/"needs_more_samples_zones.geojson").write_text(
        json.dumps({"type": "FeatureCollection", "features": []}, indent=2)
    )

sample_confidence.to_csv(CONFIDENCE_OUTPUT_DIR/"sample_confidence.csv", index=False)
recommendation_confidence.to_csv(CONFIDENCE_OUTPUT_DIR/"recommendation_confidence.csv", index=False)
(CONFIDENCE_OUTPUT_DIR/"confidence_summary.json").write_text(json.dumps(confidence_summary, indent=2))
(CONFIDENCE_OUTPUT_DIR/"uncertainty_reason_codes.json").write_text(json.dumps(reason_text, indent=2))

print("Exported Confidence/Uncertainty Layer:")
for path in sorted(CONFIDENCE_OUTPUT_DIR.iterdir()):
    print("-", path.name)


### Confidence/Uncertainty QA


In [ ]:
def run_confidence_qa():
    assert confidence_layer.shape == d22.shape == uncertainty_layer.shape
    assert np.nanmin(confidence_layer) >= 0 and np.nanmax(confidence_layer) <= 100
    assert np.nanmin(uncertainty_layer) >= 0 and np.nanmax(uncertainty_layer) <= 1
    assert np.allclose(confidence_layer[valid_map], 100*(1-uncertainty_layer[valid_map]), atol=1e-4)
    assert not needs_more_samples[~valid_map].any()
    assert len(sample_confidence) == len(inside_points)
    assert len(recommendation_confidence) == len(recommendations)
    assert sample_confidence.confidence_score.notna().all()
    assert recommendation_confidence.confidence_score.notna().all()

    near = valid_map & (distance_to_sample_m <= np.nanpercentile(distance_to_sample_m[valid_map], 25))
    far = valid_map & (distance_to_sample_m >= np.nanpercentile(distance_to_sample_m[valid_map], 75))
    near_confidence = float(np.nanmedian(confidence_layer[near]))
    far_confidence = float(np.nanmedian(confidence_layer[far]))

    return {
        "status": "PASS",
        "method": "heuristic_v1_not_calibrated",
        "valid_pixels": int(valid_map.sum()),
        "real_observations": int(len(inside_points)),
        "recommendations_scored": int(len(recommendations)),
        "needs_more_samples_zones": int(len(zones)),
        "near_sample_median_confidence": round(near_confidence, 2),
        "far_sample_median_confidence": round(far_confidence, 2),
        "far_confidence_lower_than_near": bool(far_confidence < near_confidence),
        "confidence_distribution": {
            "high_percent": confidence_summary["high_confidence_percent"],
            "moderate_percent": confidence_summary["moderate_confidence_percent"],
            "low_percent": confidence_summary["low_confidence_percent"],
        },
    }


confidence_qa = run_confidence_qa()
print(json.dumps(confidence_qa, indent=2))
(CONFIDENCE_OUTPUT_DIR/"confidence_qa.json").write_text(json.dumps(confidence_qa, indent=2))


## What to send back after testing

Please send:

1. The weight-sensitivity summary JSON and stability plot.
2. The confidence summary JSON.
3. The three-panel confidence map.
4. The existing-observation and recommendation confidence tables.
5. The final confidence QA JSON.

If weight sensitivity is fragile or far-from-sample confidence is not lower than near-sample confidence, the model should be revised before frontend integration.
